## Step 1 - Shot detection (TransNetV2)

GPU notebook. Detects shot boundaries for every video and saves them as
`SceneJSON/<batch>/<video_id>.json` (a list of `[start_frame, end_frame]`).

**Input dataset**: raw videos under `<VIDEOS_DIR>/Videos_<batch>/video/<batch>_<video_id>.mp4`.
**Output**: `./SceneJSON/` (save as a Kaggle dataset to feed Step 2).

In [ ]:
!pip install ffmpeg-python pillow
!git clone https://github.com/soCzech/TransNetV2.git
%cd TransNetV2/inference

In [ ]:
import os
import json
from tqdm import tqdm
from transnetv2 import TransNetV2

In [ ]:
# Quét đệ quy MỌI file video, không phụ thuộc cách chia thư mục
# (Videos_L21/video/..., Videos_L25_a/..., Videos_L25_b/..., lồng sâu tùy ý).
# video_name = tên file (stem), ví dụ L25_V001.mp4 -> video_name 'L25_V001',
# batch 'L25', video_id 'V001'. Khớp key join của các bước sau.
#
# GỘP THEO TÊN FILE, KHÔNG THEO FOLDER: các folder Videos_L25_a / _a1 / _b đều
# gộp về batch 'L25' vì key lấy từ TÊN FILE. Vài dataset upload LẶP một part
# (vd Videos_L25_a và Videos_L25_a1 chứa cùng dải L25_V001..V088). Ta phân biệt:
#   * cùng tên + CÙNG dung lượng  -> bản sao dư thừa, bỏ qua (giữ bản gặp đầu).
#   * cùng tên + KHÁC dung lượng  -> hai video khác nội dung xài chung id -> DỪNG,
#     vì gộp về cùng batch sẽ mất data (cần đưa hậu tố folder vào khóa).
from pathlib import Path
VIDEO_EXTS = ('.mp4', '.mkv', '.webm', '.avi', '.mov')

all_video_paths = {}
seen = {}                 # video_name -> (path, size) của bản gặp đầu tiên
conflicts = []            # trùng tên NHƯNG khác nội dung (khác size)
redundant = 0             # trùng tên VÀ cùng size -> bản sao, bỏ qua an toàn
skipped = 0
for vid in sorted(Path(VIDEOS_DIR).rglob('*')):
    if not (vid.is_file() and vid.suffix.lower() in VIDEO_EXTS):
        continue
    video_name = vid.stem                              # 'L25_V001'
    if '_' not in video_name:                          # tên không có batch -> bỏ
        skipped += 1
        continue
    if video_name in seen:
        prev_path, prev_size = seen[video_name]
        if vid.stat().st_size == prev_size:            # cùng tên + cùng size -> bản sao
            redundant += 1
        else:                                          # cùng tên, khác nội dung -> xung đột
            conflicts.append((video_name, prev_path, str(vid)))
        continue
    seen[video_name] = (str(vid), vid.stat().st_size)
    batch, video_id = video_name.split('_', 1)         # 'L25', 'V001'
    all_video_paths.setdefault(batch, {})[video_id] = str(vid)

print('batches:', {k: len(v) for k, v in all_video_paths.items()})
if skipped:
    print(f'[WARN] bỏ qua {skipped} file không có batch trong tên (dạng L25_V001 mới hợp lệ)')
if redundant:
    print(f'[WARN] bỏ qua {redundant} file trùng-lặp (cùng tên & cùng dung lượng ở '
          'folder khác, vd Videos_L25_a vs Videos_L25_a1) -> giữ bản gặp đầu tiên.')
if conflicts:
    print(f'[LỖI] {len(conflicts)} tên video trùng NHƯNG khác nội dung (khác dung lượng):')
    for name, a, b in conflicts[:20]:
        print(f'  {name}: {a}  <->  {b}')
    raise SystemExit(
        f'{len(conflicts)} video khác nội dung dùng chung tên -> video_id không '
        'duy nhất trong batch. Cần đưa hậu tố folder (_a/_b) vào khóa và sửa đồng '
        'bộ các bước sau. Xem danh sách ở trên.'
    )


In [ ]:
model = TransNetV2()
os.makedirs(SAVE_DIR, exist_ok=True)

for batch, videos in tqdm(all_video_paths.items()):
    os.makedirs(f'{SAVE_DIR}/{batch}', exist_ok=True)
    for video_id in tqdm(sorted(videos), desc=batch):
        out_path = f'{SAVE_DIR}/{batch}/{video_id}.json'
        if os.path.exists(out_path):
            continue
        _, single_frame_predictions, _ = model.predict_video(videos[video_id])
        scenes = model.predictions_to_scenes(single_frame_predictions)   # [[start, end], ...]
        with open(out_path, 'w') as f:
            json.dump(scenes.tolist(), f)

In [ ]:
# Clean up the cloned repo so it is not exported with the dataset
!rm -rf ./TransNetV2 2>/dev/null
%cd /kaggle/working